# Kaggle コミットで Hardcore 追加学習を回すテンプレ

Basic で学習済みのベストモデルを W&B から取得し、Hardcore（`BipedalWalkerHardcore-v3`）へ追加学習（カリキュラム学習）するノートです。sweep 用の `kaggle_commit.ipynb` と同じ準備手順（clone → pip install → W&B ログイン）で、最後のセルだけが違います。

**⚠️ 2026-07-14 の変更: このノートは編集不要になりました**

- 実行する設定（resume 先・config・seed・ステップ数）の既定値は、実行時に clone される repo の **`configs/hardcore_next_run.yaml`** から読みます。次の周の内容は PR でそのファイルを更新するので、**Kaggle 側のノートは Save & Run All するだけ**で常に最新の設定で走ります。
- **この仕組みへの切り替えのため、Kaggle 側のノートを今回1回だけ GitHub から再インポートしてください**（File → Import Notebook → GitHub で `notebooks/kaggle_hardcore_finetune.ipynb` を指定）。以後の再インポートは不要です。
- 背景: 以前は実行変数がノート内に埋まっており、GitHub を更新しても Kaggle 側のノートが古いまま実行されて、前と同じ設定で1周走ってしまう事故が起きました。

**注意事項（読んでから実行）**

- このノートは **CPU セッション** で使う。GPU は選択しない。
- **インターネット接続を有効化** すること（Settings → Internet を On。電話番号認証が必要）。
- W&B の API キーは Add-ons → Secrets に `WANDB_API_KEY` として登録し、チェックボックスを On にする。
- 実測で 100万ステップ ≒ 8〜10 時間。セッション上限（約12時間）に収まるが、**途中で切られると checkpoint も消える**ので、心配なら `TOTAL_TIMESTEPS` を上書きして減らし複数セッションに分ける。
- 使い方: 右上の Save Version → 「**Save & Run All（Commit）**」で放置実行する。
- 学習が終わると、ベストモデルの走りの動画が **自動で W&B の run にアップロードされる**（train.py の機能。追加操作は不要）。


In [ ]:
# 1. リポジトリを clone
# 【なぜ】Kaggle が用意するマシンはまっさらで、私たちのコードが入っていない。
#         まず GitHub からコード一式をこのマシンにダウンロードする。
# 【成功すると】SB3-Team フォルダができて src/train.py などが使える状態になり、
#              %cd でそのフォルダに移動する。
!git clone https://github.com/shironaganegi/SB3-Team.git
%cd SB3-Team

In [ ]:
# 2. 依存ライブラリをインストール
# 【なぜ】学習コードは PyTorch・gymnasium・sb3-contrib などに依存しているが、
#         まっさらなマシンには入っていない（または版が合わない）ため。
!pip install -r requirements.txt

In [ ]:
# 3. W&B にログイン
# 【なぜ】このマシンから W&B のモデルファイルを取得し、学習結果を送るため。
#         キーは Add-ons → Secrets の WANDB_API_KEY から読む（直書き禁止）。
from kaggle_secrets import UserSecretsClient
import os

user_secrets = UserSecretsClient()
os.environ["WANDB_API_KEY"] = user_secrets.get_secret("WANDB_API_KEY")
!wandb login --relogin $WANDB_API_KEY

In [ ]:
# 4. ベースモデルを W&B から取得し、Hardcore へ追加学習（このノートの本体）
# 【なぜ】Hardcore をゼロから学習するのは CPU ではほぼ無理なので、
#         「歩ける」Basic モデルに障害物対応だけを追加学習させる（docs/ROADMAP.md Step 4a/4b）。
# 【成功すると】W&B に hardcore の run が1本増え、学習が終わると
#              モデル(zip)が run の Files にアップロードされる。
#
# 【このセルの仕組み】実行する設定の既定値は、clone した repo の
#   configs/hardcore_next_run.yaml から読む（次の周の内容は PR で更新される）。
#   なので普段は **何も書き換えずに Save & Run All するだけでよい**。
#   個人で別の設定を試したいときだけ、下の変数に値を入れて上書きする。

RESUME_RUN_PATH = ""      # 例: "sai3desuyo-/bipedal-timetrial/xsrplaip"。空なら repo の既定値
CONFIG_PATH = ""          # 例: "configs/hardcore_finetune.yaml"。空なら repo の既定値
SEED = None               # 例: 1。None なら repo の既定値（人ごとに変えると探索が広がる）
TOTAL_TIMESTEPS = None    # 例: 500_000。None なら repo の既定値

import wandb, yaml

# --- repo の既定値を読み、上書きがあれば反映 ---
defaults = yaml.safe_load(open("configs/hardcore_next_run.yaml"))
resume_run_path = RESUME_RUN_PATH or defaults["resume_run_path"]
config_path     = CONFIG_PATH or defaults["config_path"]
seed            = SEED if SEED is not None else defaults["seed"]
total_timesteps = TOTAL_TIMESTEPS if TOTAL_TIMESTEPS is not None else defaults["total_timesteps"]

# --- ベースモデルのダウンロード ---
api = wandb.Api()
base_run = api.run(resume_run_path)
model_file = [f for f in base_run.files() if f.name == "model.zip"]
assert model_file, f"{resume_run_path} に model.zip がありません（Finished した run を指定してください）"
model_file[0].download(root="base_model", replace=True)

# --- config のコピーを作る（use_wandb を有効化し、上の値を反映）---
cfg = yaml.safe_load(open(config_path))
cfg["use_wandb"] = True
cfg["seed"] = seed
cfg["total_timesteps"] = total_timesteps
yaml.safe_dump(cfg, open("hardcore_kaggle.yaml", "w"), allow_unicode=True)

# --- これから走る設定を確認用に表示（意図と違ったらここで気づけるように）---
print("=" * 60)
print("これから走る設定:")
print(f"  resume 元      : {resume_run_path}（{base_run.name}: eval_reward={base_run.summary.get('eval/mean_reward')}）")
print(f"  config         : {config_path}")
print(f"  vel_coef       : {cfg.get('vel_coef')}")
print(f"  seed           : {seed}")
print(f"  total_timesteps: {total_timesteps}")
print("=" * 60)

# --- 追加学習を実行 ---
!python src/train.py --config hardcore_kaggle.yaml --resume base_model/model.zip